In [72]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import LabelEncoder

## Models Notebook:

In [2]:
# import data
master_chembl = pd.read_csv('data/chembl_export_final.csv')

In [3]:
master_chembl.head()

,canonical_smiles,molecule_chembl_id,pchembl_value,value,receptor_class,MolWt,LogP,NumHDonors,NumHAcceptors,TPSA,NumRotatableBonds,RingCount,FractionCSP3,MolRefractory
0,c1ccc(CNCCOc2cccc3[nH]ncc23)cc1,CHEMBL87137,6.50,315.5,Dopamine_D2,267.332,2.73150,2,3,49.94,6,3,0.187500,79.7374
1,O=c1c(CO)coc2cc(OCCCN3CCC(c4noc5cc(F)ccc45)CC3...,CHEMBL151475,6.75,179.0,Dopamine_D2,452.482,4.21410,1,7,88.94,7,5,0.360000,120.9008
2,O=C(NC/C=C/CN1CCN(c2cccc(Cl)c2Cl)CC1)c1ccc(-c2...,CHEMBL180010,6.43,375.0,Dopamine_D2,481.427,5.16360,1,4,48.47,7,4,0.230769,136.2612
3,COc1ccccc1N1CCN(CC(=O)Nc2cccc(C)c2)CC1,CHEMBL184486,9.15,0.7,Dopamine_D2,339.439,2.76432,1,4,44.81,5,3,0.350000,101.4557
4,CCCN(CCC)C1CC=C(C#C[Si](C)(C)C)CC1,CHEMBL163087,7.27,54.0,Dopamine_D2,277.528,4.46810,0,1,3.24,5,1,0.764706,88.9990


In [4]:
master_chembl.columns

Index(['canonical_smiles', 'molecule_chembl_id', 'pchembl_value', 'value',
       'receptor_class', 'MolWt', 'LogP', 'NumHDonors', 'NumHAcceptors',
       'TPSA', 'NumRotatableBonds', 'RingCount', 'FractionCSP3',
       'MolRefractory'],
      dtype='str')

In [35]:
# Feature isolations into X, y df's
feature_cols = ["MolWt", "LogP", "NumHDonors", "NumHAcceptors", "TPSA", "NumRotatableBonds", "RingCount", "FractionCSP3", "MolRefractory"]
X = master_chembl[feature_cols]

y_true = master_chembl["receptor_class"]

In [36]:
# Standardize features (vital since descriptors are on vastly different scales, e.g., MolWt vs LogP)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled

array([[-1.52345292, -0.85334276,  0.46349325, ..., -1.06522767,
        -0.89121402, -1.36803656],
       [ 0.34937156,  0.27179099, -0.47595653, ...,  0.9289544 ,
         0.27191014,  0.18035171],
       [ 0.64215528,  0.99235925, -0.47595653, ..., -0.06813663,
        -0.59946047,  0.75814323],
       ...,
       [ 0.80356311,  0.2872724 ,  0.46349325, ..., -0.06813663,
        -0.46979222, -0.24902723],
       [-1.10908481, -0.809327  , -0.47595653, ..., -0.06813663,
        -1.73405761, -1.25662651],
       [ 1.0894685 ,  0.89021235, -0.47595653, ...,  1.92604544,
         0.37305137,  1.38444389]], shape=(3995, 9))

## PCA

In [80]:
"""
https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html
"""

#### Reducing 9 descriptors to 4 principal components
pca = PCA(n_components= 4, random_state=42)

X_pca = pca.fit_transform(X_scaled)

print(f"Explained variance ratio sum (4 components): {pca.explained_variance_ratio_.sum():.4f}")

Explained variance ratio sum (4 components): 0.8879


### Explained Variance Ratio Results:
* Explained variance ratio sum (3 components): 0.8042
* Explained variance ratio sum (4 components): 0.8879
* Explained variance ratio sum (5 components): 0.9522
* Explained variance ratio sum (6 components): 0.9739
* Explained variance ratio sum (7 components): 0.9880
* Explained variance ratio sum (8 components): 0.9971
  
I will proceed with 4 principle components to keep dimensionality down for K-means while capturing 89% of the variance.

In [82]:
# Create a dataframe of the component loadings
# pca.components_ has the weights, feature_cols has the original RDKit names
pca_results = pd.DataFrame(
    pca.components_,
    columns = feature_cols,
    index=[f'PC{i+1}' for i in range(pca.n_components_)]
)

# Display the pca_results table
print(pca_results)

        MolWt      LogP  NumHDonors  NumHAcceptors      TPSA  \
PC1  0.482867  0.220128    0.115861       0.367124  0.309662   
PC2 -0.098957 -0.445129    0.540605       0.292315  0.543923   
PC3  0.000883 -0.435801   -0.342324       0.230151 -0.039181   
PC4 -0.023234 -0.370437   -0.183230       0.147726  0.095989   

     NumRotatableBonds  RingCount  FractionCSP3  MolRefractory  
PC1           0.401020   0.285417      0.082998       0.474468  
PC2          -0.063309  -0.240448     -0.155059      -0.180684  
PC3           0.242651  -0.267760      0.711692      -0.036140  
PC4          -0.470683   0.754035      0.086357      -0.007576  


### PCA Analysis
* PC1: This component is dominated by molecular weight and the molecule's polarizability and bulk volume, meaning large vs small molecules.
* PC2: This component is dominated by number of hydrogen donors and TPSA which estimates the surface area of the molecule's polar atoms, meaning hydrophilic vs lipophylic molecules
* PC3: This component is dominated by FractionCSP3, which is used as an indicator of molecular three-dimensionality and complexity. We can interpret this as complex three-dimensional molecules vs more flat ring-like structures. 
* PC4: This component is dominated by ring count vs number of rotatable bonds which is similar to PC3 in that on one end we have ring-based molecules but on the other we now have flexible configurations arising from the number of rotatable bonds

## K-Means Clustering

In [84]:
"""
https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html
""" 

# 4 clusters for 4 unique classes
kmeans = KMeans(
    n_clusters=4, 
    random_state=42, 
    init='k-means++',
    max_iter=300,
    n_init=10 # number of runs to find more optimal final centroid location
)

cluster_labels = kmeans.fit_predict(X_pca)
master_chembl["cluster"] = cluster_labels

## Evaluation

In [85]:
contingency_table = pd.crosstab(y_true, master_chembl["cluster"])
print("Contingency Table (Receptor Classes vs K-Means Clusters):")
print(contingency_table)

le = LabelEncoder()
y_true_encoded = le.fit_transform(y_true)
ari = adjusted_rand_score(y_true_encoded, cluster_labels)

print("\n")
print(f"Adjusted Rand Index (ARI): {ari:.4f}")

Contingency Table (Receptor Classes vs K-Means Clusters):
cluster            0    1   2    3
receptor_class                    
Dopamine_D2      806   71  26   92
GABA_A            63  586   1  350
Histamine_H1     107  477   0  416
Serotonin_5HT2A  415  109   0  476


Adjusted Rand Index (ARI): 0.1933


### Observations and Conclusions:
It wasn't possible to segment the data back into the four receptor classes based on the four PCA components. The k-means algorithm did not segment based on neuromolecular target but based on structural (dis)similarities.
  
* Cluster 0 represents the molecular weight/size component which shows that dopamine class segments well into this cluster. 
* The GABA class is more highly clusterd in PCA 2 and 4 for hydrophilicity vs lipophylicity and ring count vs rotatable bonds.
* The FractionCSP3 dominated cluster has very few candidates total and of the data present in this cluster, a vast majority of dopamine ligand molecules, suggesting that higher molecular weight/size correlates with three-dimensional complexity in this class.
* Finally, the fourth cluster which includes the ring count vs rotatable bonds component shows a strong presence from the GABA, histamine and serotonin class.
  
### Future Directions:
It would be interesting to review the generalities of the biology and chemical structures of the ligands of each neuro-receptor class and to compare them to the results. It would also be interesting to run various algorithms using Morgan Fingerprints which can turn molecular and chemical substructures into features!